In [1]:
import os
import os
from dotenv import load_dotenv
load_dotenv()
token = os.getenv("HF_TOKEN")
#allows me to download gated model files, private repos and everything related to running tabular pfn.


In [2]:
from tabpfn import TabPFNRegressor #class def
model = TabPFNRegressor() #instantiate and loads pretrained TabPFN
print("loaded")

loaded


In [3]:
import sys
!{sys.executable} -m pip install -U pip
!{sys.executable} -m pip install tabpfn #Installs tabpfn


In [4]:
import numpy as np
from wfcrl import environments as envs #grabs wind turbine environments

env = envs.make("Turb3_Row1_Floris", max_num_steps=70) #take random environment set up 
obs = env.reset() #New epsiode to get initial observations from episode 


In [5]:
def get_layout_xy(env):
    for a in ["farm_case", "config", "case"]:
        if hasattr(env, a):
            c = getattr(env, a)
            if hasattr(c, "xcoords") and hasattr(c, "ycoords"):
                return np.asarray(c.xcoords, float), np.asarray(c.ycoords, float)
    if hasattr(env, "interface") and hasattr(env.interface, "config"):
        c = env.interface.config
        if hasattr(c, "xcoords") and hasattr(c, "ycoords"):
            return np.asarray(c.xcoords, float), np.asarray(c.ycoords, float)
    raise RuntimeError("Could not find turbine coordinates on env")

def bearing_deg(dx, dy):
    ang = (np.degrees(np.arctan2(dx, dy)) + 360.0) % 360.0 #convert vector into compass based angles 
    return ang

def angle_diff_deg(a, b): #compute smallest diff between angles a & b
    d = (a - b + 180.0) % 360.0 - 180.0
    return np.abs(d) #return magnitude so always positive 

x, y = get_layout_xy(env) #Gets turbine layout
n = env.num_turbines #number of nodes in the environment

yaw = np.asarray(obs["yaw"], float) #Yaw angle for wfcrl obs.
ws = np.asarray(obs["wind_speed"], float) #wind sped 
wd = np.asarray(obs["wind_direction"], float) #wind direction

node_x = np.column_stack([yaw, ws, wd]).astype(np.float32) #Node feature matrix shape

#Topology encoding starts HERE
src, dst = [], [] #edge lists from graph
dist_thresh = 1200.0 #turbine distance thresh.
cone = 45.0 #downwind turbine

for i in range(n): #go through all wind turbines as upstream
    downwind = (wd[i] + 180.0) % 360.0 #wake direction
    for j in range(n): #all turbines as downstream targets
        if i == j:
            continue
        dx, dy = x[j] - x[i], y[j] - y[i] #no self-edge
        d = float(np.hypot(dx, dy)) #positions from i to j
        if d > dist_thresh: #too far
            continue
        b = bearing_deg(dx, dy) #if j downwind of i add directed edge
        if angle_diff_deg(b, downwind) <= cone: #within cone threshold
            src.append(i)
            dst.append(j)

edge_index = np.array([src, dst], dtype=np.int64) #graph topology in (2, E) format

graph = {"x": node_x, "edge_index": edge_index} #graph object (features + topology)
graph["x"].shape, graph["edge_index"].shape #check nodes by features, edge dimensions.


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/gymnasium/core.py:311: UserWarning: WARN: env.farm_case to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.farm_case` for environment variables or `env.get_wrapper_attr('farm_case')` that will search the reminding wrappers.
  logger.warn(


((3, 3), (2, 3))

In [6]:
import numpy as np
from tabpfn import TabPFNRegressor

model = TabPFNRegressor()
X_train = [graph for _ in range(8)]
y_train = np.random.randn(8).astype(np.float32)

try:
    model.fit(X_train, y_train) #Breaking code line since since TabPFN expects 2d numeric array, but passing in dicts of topology
except Exception as e:
    print(type(e).__name__)
    print(str(e).splitlines()[0])
    print("Conclusion: TabPFN expects a 2D tabular feature matrix; graph dict inputs (x, edge_index) are rejected.")



TabPFNValidationError
Expected 2D array, got 1D array instead:
Conclusion: TabPFN expects a 2D tabular feature matrix; graph dict inputs (x, edge_index) are rejected.


In [7]:
#This test makes it work by converting to tabular, but 

import numpy as np
from tabpfn import TabPFNRegressor

env = envs.make("Turb3_Row1_Floris", max_num_steps=70)
out = env.reset()
obs0 = out[0] if isinstance(out, tuple) else out

def obs_to_graph(obs, env):
    x, y = get_layout_xy(env)
    n = env.num_turbines

    yaw = np.asarray(obs["yaw"], float)
    ws  = np.asarray(obs["wind_speed"], float)
    wd  = np.asarray(obs["wind_direction"], float)
    node_x = np.column_stack([yaw, ws, wd]).astype(np.float32)

    src, dst = [], [] #same process as above cell for production of topology relationship
    dist_thresh = 1200.0
    cone = 45.0
    for i in range(n):
        downwind = (wd[i] + 180.0) % 360.0
        for j in range(n):
            if i == j:
                continue
            dx, dy = x[j] - x[i], y[j] - y[i]
            d = float(np.hypot(dx, dy))
            if d > dist_thresh:
                continue
            b = bearing_deg(dx, dy)
            if angle_diff_deg(b, downwind) <= cone:
                src.append(i); dst.append(j)

    edge_index = np.array([src, dst], dtype=np.int64)
    return {"x": node_x, "edge_index": edge_index}

#Define random actions to take within the graphical model.
def random_action(env):
    n = env.num_turbines
    return {"yaw": np.random.uniform(-5, 5, size=n).astype(np.float32)}

graphs = []
targets = []

obs = obs0
for t in range(30):
    g = obs_to_graph(obs, env)
    graphs.append(g)
    targets.append(float(g["x"][:, 1].sum()))  # example target: sum wind_speed

    obs, reward, termination, truncation, info = env.step(random_action(env))
    if termination or truncation:
        obs = env.reset()
        obs = obs[0] if isinstance(obs, tuple) else obs

X_tab = np.stack([g["x"].reshape(-1) for g in graphs], axis=0)  #flattens each sample
y = np.array(targets, dtype=np.float32) #stack samples to n_sampels, n_turbines * 3

#Topology is dropped, so edges never passed TabPFN
#Turbine orering is hard coded 
#variable size lost
#lost relational bias

model = TabPFNRegressor()
model.fit(X_tab, y)
print("Tabular fit OK. Shape:", X_tab.shape) #succeeds to run by making X_tab 2D, but topology is essentially ignored 




/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/gymnasium/core.py:311: UserWarning: WARN: env.farm_case to get variables from other wrappers is deprecated and will be removed in v1.0, to get this variable you can do `env.unwrapped.farm_case` for environment variables or `env.get_wrapper_attr('farm_case')` that will search the reminding wrappers.
  logger.warn(


Tabular fit OK. Shape: (30, 9)


In [8]:
try:
    TabPFNRegressor().fit(graphs, y)
except Exception as e:
    print("Graph fit failed as expected:", type(e).__name__)


Graph fit failed as expected: TabPFNValidationError


In [9]:
model = TabPFNRegressor()
try:
    model.fit(graphs, y)
except Exception as e:
    print(type(e).__name__)
    print("Graph input rejected.")


TabPFNValidationError
Graph input rejected.


In [10]:
model = TabPFNRegressor()
model.fit(X_tab, y)

print("X_tab shape:", X_tab.shape)
print("First row (flattened features):")
print(X_tab[0])

print("First row reshaped back to (n_turbines, n_features_per_turbine):")
print(X_tab[0].reshape(env.num_turbines, -1))

print("Column meaning for first row (per turbine):")
for t in range(env.num_turbines):
    yaw, ws, wd = X_tab[0].reshape(env.num_turbines, -1)[t]
    print(f"T{t+1}: yaw={yaw:.3f}, wind_speed={ws:.3f}, wind_dir={wd:.3f}")

pred = model.predict(X_tab)
print("Predictions (first 10):", pred[:10])
print("Targets (first 10):     ", y[:10])
print("MSE:", float(np.mean((pred - y) ** 2)))



X_tab shape: (30, 9)
First row (flattened features):
[  0.          7.53387   282.591       0.          7.2228627 282.72192
   0.          7.2201385 282.77084  ]
First row reshaped back to (n_turbines, n_features_per_turbine):
[[  0.          7.53387   282.591    ]
 [  0.          7.2228627 282.72192  ]
 [  0.          7.2201385 282.77084  ]]
Column meaning for first row (per turbine):
T1: yaw=0.000, wind_speed=7.534, wind_dir=282.591
T2: yaw=0.000, wind_speed=7.223, wind_dir=282.722
T3: yaw=0.000, wind_speed=7.220, wind_dir=282.771
Predictions (first 10): [21.976816 21.883291 21.859104 21.839657 21.84175  21.857405 21.741459
 21.898024 21.954115 22.048437]
Targets (first 10):      [21.976871 21.883045 21.858929 21.840511 21.841963 21.857355 21.741642
 21.897167 21.954063 22.048573]
MSE: 2.008439565770459e-07
